In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pointbiserialr

pio.templates.default = "plotly_dark"

In [2]:
df_healthy = pd.read_parquet("../../data/processed/dataset_features.parquet")


In [3]:
pd.set_option("display.max_seq_items", None)
df_healthy.columns

Index(['degree_AF3', 'clustering_AF3', 'betweenness_AF3', 'hub_frequency_AF3',
       'sd_degree_AF3', 'bin_clustering_AF3', 'degree_AF4', 'clustering_AF4',
       'betweenness_AF4', 'hub_frequency_AF4', 'sd_degree_AF4',
       'bin_clustering_AF4', 'degree_AF8', 'clustering_AF8', 'betweenness_AF8',
       'hub_frequency_AF8', 'sd_degree_AF8', 'bin_clustering_AF8',
       'degree_AFz', 'clustering_AFz', 'betweenness_AFz', 'hub_frequency_AFz',
       'sd_degree_AFz', 'bin_clustering_AFz', 'degree_C1', 'clustering_C1',
       'betweenness_C1', 'hub_frequency_C1', 'sd_degree_C1',
       'bin_clustering_C1', 'degree_C2', 'clustering_C2', 'betweenness_C2',
       'hub_frequency_C2', 'sd_degree_C2', 'bin_clustering_C2', 'degree_C3',
       'clustering_C3', 'betweenness_C3', 'hub_frequency_C3', 'sd_degree_C3',
       'bin_clustering_C3', 'degree_C4', 'clustering_C4', 'betweenness_C4',
       'hub_frequency_C4', 'sd_degree_C4', 'bin_clustering_C4', 'degree_C5',
       'clustering_C5', 'between

In [4]:

print(df_healthy["age"].value_counts())
print(df_healthy["gender"].value_counts())
print(df_healthy["handedness"].value_counts())
print(df_healthy["education"].value_counts())
print(df_healthy["drug"].value_counts())

# garantir tipos
df_healthy["age"] = df_healthy["age"].astype(str)
df_healthy["gender"] = df_healthy["gender"].astype(str)
df_healthy["handedness"] = df_healthy["handedness"].astype(str)
df_healthy["education"] = df_healthy["education"].astype(str)
df_healthy["drug"] = df_healthy["drug"].astype(str)


age
20-25    36
25-30    28
60-65    12
70-75    12
65-70    10
30-35     6
55-60     2
75-80     2
Name: count, dtype: int64
gender
2    84
1    24
Name: count, dtype: int64
handedness
right          96
left           10
ambedextor      2
Name: count, dtype: int64
education
gymnasium     80
realschule    26
gymansium      2
Name: count, dtype: int64
drug
0.0    108
Name: count, dtype: int64


In [5]:
# Organizando as variáveis (dicotomia alinhada ao LEMON: young ~20–35, older ~59–77+)
df_healthy["age_group"] = df_healthy["age"].replace({
    "20-25": "young",
    "25-30": "young",
    "30-35": "young",
    "35-40": "young",
    "55-60": "older",
    "60-65": "older",
    "65-70": "older",
    "70-75": "older",
    "75-80": "older",
})

# Calculando as médias das métricas
degree_cols = [c for c in df_healthy.columns if c.startswith("degree_")]
clustering_cols = [c for c in df_healthy.columns if c.startswith("clustering_")]
betweenness_cols = [c for c in df_healthy.columns if c.startswith("betweenness_")]
sd_degree_cols = [c for c in df_healthy.columns if c.startswith("sd_degree_")]
bin_clustering_cols = [c for c in df_healthy.columns if c.startswith("bin_clustering_")]
df_healthy["degree_mean"] = df_healthy[degree_cols].mean(axis=1)
df_healthy["clustering_mean"] = df_healthy[clustering_cols].mean(axis=1)
df_healthy["betweenness_mean"] = df_healthy[betweenness_cols].mean(axis=1)
df_healthy["sd_degree_mean"] = df_healthy[sd_degree_cols].mean(axis=1)
df_healthy["bin_clustering_mean"] = df_healthy[bin_clustering_cols].mean(axis=1)


In [6]:
df_healthy["age"].value_counts()

age
20-25    36
25-30    28
60-65    12
70-75    12
65-70    10
30-35     6
55-60     2
75-80     2
Name: count, dtype: int64

In [7]:

GRAPH_METRICS = {
    "degree_mean": "Grau médio por janela (normalizado)",
    "sd_degree_mean": "Variabilidade temporal do grau (DP)",
    "bin_clustering_mean": "Agrupamento binário por janela",
    "clustering_mean": "Agrupamento ponderado (REA)",
    "betweenness_mean": "Intermediação (REA, distância 1/peso)",
}

# a célula seguinte renomeia age_group/gender; aceita os dois nomes para permitir reexecução
age_col = "age_group" if "age_group" in df_healthy.columns else "Faixa Etária"
gender_col = "gender" if "gender" in df_healthy.columns else "Gênero"

sample_summary = (
    df_healthy[["subject_id", age_col, gender_col]]
    .drop_duplicates()
    .groupby([age_col, gender_col])
    .size()
    .reset_index(name="n")
)

sample_summary

,age_group,gender,n
0,older,1,4
1,older,2,15
2,young,1,8
3,young,2,27


In [8]:
df_healthy = (
    df_healthy[df_healthy["condition"] != "E0"]
    .rename(columns={"age_group": "Faixa Etária", "gender": "Gênero"})
    .copy()
)

df_healthy[["Faixa Etária", "Gênero"]].value_counts()

Faixa Etária  Gênero
young         2         54
older         2         30
young         1         16
older         1          8
Name: count, dtype: int64

### Idade

In [9]:
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Faixa Etária",
        y=metric,
        color="Faixa Etária",
        title=f"{label} vs Age Group",
        points='all',
        labels={
            "Faixa Etária": "Faixa Etária",
            metric: label
        }
    )
    fig.update_layout(
        xaxis_title="Faixa Etária",
        yaxis_title=label,
        title=f"{label} vs Faixa Etária"
    )
    fig.show()

In [10]:
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

AGE_PANELS = [
    ("degree_mean", "Grau médio normalizado", "A"),
    ("bin_clustering_mean", "Coeficiente de Agrupamento", "B"),
    ("betweenness_mean", "Centralidade de Intermediação (REA)", "C"),
]
AGE_LABELS = {"young": "Adultos jovens", "older": "Adultos mais velhos"}
AGE_COLORS = {"Adultos jovens": "#636EFA", "Adultos mais velhos": "#EF553B"}

# um ponto por participante: média de EO e EC
per_participant = (
    df_healthy.groupby(["subject_id", "Faixa Etária"], as_index=False)[[m for m, _, _ in AGE_PANELS]]
    .mean()
    .assign(grupo=lambda d: d["Faixa Etária"].map(AGE_LABELS))
)

fig_age = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.09,
    subplot_titles=[f"<b>{tag}</b>" for _, _, tag in AGE_PANELS],
)
for col, (metric, ylabel, _tag) in enumerate(AGE_PANELS, start=1):
    for grupo, color in AGE_COLORS.items():
        sub = per_participant.loc[per_participant["grupo"] == grupo]
        fig_age.add_trace(
            go.Box(
                x=sub["grupo"],
                y=sub[metric],
                name=grupo,
                marker=dict(color=color, size=9, opacity=0.8),
                line=dict(color=color, width=2.5),
                boxpoints="all",
                jitter=0.35,
                pointpos=0,
                showlegend=False,
            ),
            row=1,
            col=col,
        )
    fig_age.update_yaxes(title_text=ylabel, row=1, col=col)

fig_age.update_xaxes(categoryorder="array", categoryarray=list(AGE_COLORS), tickfont_size=18)
fig_age.update_yaxes(title_font_size=20, tickfont_size=16)
fig_age.update_annotations(font_size=26, x=None, xanchor="left")
for ann, col in zip(fig_age.layout.annotations, range(1, 4)):
    ann.x = fig_age.get_subplot(1, col).xaxis.domain[0]
fig_age.update_layout(
    template="plotly_white",
    font=dict(size=18, color="black"),
    width=1500,
    height=560,
    margin=dict(l=80, r=30, t=70, b=70),
)
fig_age.show()

out_png = Path("../../figures/fig_metricas_idade_abc.png")
out_png.parent.mkdir(parents=True, exist_ok=True)
try:
    fig_age.write_image(out_png, scale=3)
    print(f"Salvo: {out_png.resolve()}")
except ValueError as err:
    # kernel iniciado antes de instalar o kaleido: reinicie o kernel para exportar PNG
    out_html = out_png.with_suffix(".html")
    fig_age.write_html(out_html)
    print(f"PNG não exportado ({err.__class__.__name__}). HTML: {out_html.resolve()}")

Salvo: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\figures\fig_metricas_idade_abc.png


-------

### Condition

In [11]:

# plotly
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="condition",
        y=metric,
        color="condition",
        title=f"{metric} vs Condition",
        points="all",
        labels={
            "condition": "Condition",
            metric: metric
        }
    )
    fig.update_layout(
        xaxis_title="Condition",
        yaxis_title=label,
        title=f"{label} vs Condition"
    )
    fig.show()

------

In [12]:
# Boxplot Age Group x Condition
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Faixa Etária",
        y=metric,
        color="condition",
        title=f"{label} vs Faixa Etária",
        points="all",
        labels={
            "Faixa Etária": "Faixa Etária",
            label: label
        }
    )
    fig.update_layout(
        xaxis_title=f"{label}: EO vs EC estratificado por idade",
        yaxis_title=label,
        title=f"{label} vs Faixa Etária"
    )
    fig.show()

### Figura 6 complementar — idade × EO/EC (painéis A–C)

Mesmas métricas da Figura 6, agora com **EO vs EC** lado a lado em cada faixa etária. Layout vertical (A/B/C) para legibilidade na página, com **uma legenda** comum.

In [13]:
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

PANEL_METRICS = [
    ("degree_mean", "Grau médio por janela (normalizado)", "A"),
    ("bin_clustering_mean", "Agrupamento binário por janela", "B"),
    ("betweenness_mean", "Intermediação (REA)", "C"),
]

age_order = ["young", "older"]
cond_order = ["EO", "EC"]
color_map = {"EO": "#636EFA", "EC": "#EF553B"}

fig6 = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[f"{tag}. {lab}" for _, lab, tag in PANEL_METRICS],
)

for row, (metric, ylabel, _tag) in enumerate(PANEL_METRICS, start=1):
    for cond in cond_order:
        sub = df_healthy.loc[df_healthy["condition"] == cond]
        fig6.add_trace(
            go.Box(
                x=sub["Faixa Etária"],
                y=sub[metric],
                name=cond,
                legendgroup=cond,
                marker_color=color_map[cond],
                line_color=color_map[cond],
                offsetgroup=cond,
                boxpoints="all",
                jitter=0.25,
                pointpos=0,
                showlegend=(row == 1),
            ),
            row=row,
            col=1,
        )
    fig6.update_yaxes(title_text=ylabel, row=row, col=1)
    fig6.update_xaxes(
        categoryorder="array",
        categoryarray=age_order,
        row=row,
        col=1,
    )

fig6.update_xaxes(title_text="Faixa etária", row=3, col=1)
fig6.update_layout(
    height=900,
    width=700,
    boxmode="group",
    legend_title_text="Condição",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=90, b=60),
    title_text="Figura 6 complementar: métricas de rede por faixa etária e condição (EO/EC)",
)
fig6.for_each_annotation(lambda a: a.update(font_size=12))
fig6.show()

out_dir = Path("../../figures")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "fig6_age_eo_ec_panels.png"
try:
    fig6.write_image(str(out_path), scale=2)
    print(f"Salvo: {out_path.resolve()}")
except Exception as e:
    html_path = out_dir / "fig6_age_eo_ec_panels.html"
    fig6.write_html(str(html_path))
    print(f"PNG não exportado ({type(e).__name__}: {e}). HTML: {html_path}")


Salvo: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\figures\fig6_age_eo_ec_panels.png


------------------

### Idade: testes por participante

Um valor por participante (média de EO e EC), Mann-Whitney bilateral entre adultos mais velhos e adultos jovens, correção de Benjamini–Hochberg sobre as 5 métricas e correlação rank-biserial (negativa = menor nos adultos mais velhos). EO e EC isolados aparecem só como descrição.

In [14]:
# Idade com um valor por participante: EO e EC do mesmo sujeito não são independentes
from scipy.stats import false_discovery_control, mannwhitneyu

per_subject = df_healthy.pivot_table(
    index=["subject_id", "Faixa Etária"], columns="condition", values=list(GRAPH_METRICS)
)


def age_test(values: pd.Series) -> dict:
    values = values.dropna()
    older = values.xs("older", level="Faixa Etária")
    young = values.xs("young", level="Faixa Etária")
    u, p = mannwhitneyu(older, young, alternative="two-sided")
    return {
        "n_older": len(older),
        "n_young": len(young),
        "mediana_older": older.median(),
        "mediana_young": young.median(),
        "p_value": p,
        # negativo = menor nos adultos mais velhos
        "rank_biserial": 2 * u / (len(older) * len(young)) - 1,
    }


age_rows, interaction_rows = [], []
for metric, label in GRAPH_METRICS.items():
    eo, ec = per_subject[(metric, "EO")], per_subject[(metric, "EC")]
    age_rows.append({"metric": label, "medida": "média EO/EC", **age_test((eo + ec) / 2)})
    age_rows.append({"metric": label, "medida": "só EO (descritivo)", **age_test(eo)})
    age_rows.append({"metric": label, "medida": "só EC (descritivo)", **age_test(ec)})
    interaction_rows.append({"metric": label, "medida": "EO − EC", **age_test(eo - ec)})

age_tests = pd.DataFrame(age_rows)
main = age_tests["medida"] == "média EO/EC"
age_tests.loc[main, "p_fdr"] = false_discovery_control(age_tests.loc[main, "p_value"])
age_tests.round(4)

,metric,medida,n_older,n_young,mediana_older,mediana_young,p_value,rank_biserial,p_fdr
0,Grau médio por janela (normalizado),média EO/EC,19,35,0.5453,0.6063,0.0033,-0.4887,0.0056
1,Grau médio por janela (normalizado),só EO (descritivo),19,35,0.5050,0.5804,0.0003,-0.5970,NaN
2,Grau médio por janela (normalizado),só EC (descritivo),19,35,0.5824,0.6241,0.0372,-0.3474,NaN
3,Variabilidade temporal do grau (DP),média EO/EC,19,35,0.1978,0.2002,0.0526,-0.3233,0.0658
4,Variabilidade temporal do grau (DP),só EO (descritivo),19,35,0.1928,0.2021,0.0056,-0.4617,NaN
5,Variabilidade temporal do grau (DP),só EC (descritivo),19,35,0.1970,0.2009,0.2692,-0.1850,NaN
6,Agrupamento binário por janela,média EO/EC,19,35,0.7216,0.7677,0.0033,-0.4887,0.0056
7,Agrupamento binário por janela,só EO (descritivo),19,35,0.7007,0.7523,0.0003,-0.6060,NaN
8,Agrupamento binário por janela,só EC (descritivo),19,35,0.7406,0.7752,0.0297,-0.3624,NaN
9,Agrupamento ponderado (REA),média EO/EC,19,35,0.5285,0.5911,0.0032,-0.4917,0.0056


In [28]:
tabela = (
    age_tests[age_tests["medida"] == "média EO/EC"]
    .set_index("metric")[["mediana_young", "mediana_older", "p_fdr", "rank_biserial"]]
    .rename(columns={
        "mediana_young": "Adultos jovens (n = 35), mediana",
        "mediana_older": "Adultos mais velhos (n = 19), mediana",
        "p_fdr": "p_FDR",
        "rank_biserial": "Efeito rank-biserial",
    })
)
tabela.style.format({
    "Adultos jovens (n = 35), mediana": "{:.4g}",
    "Adultos mais velhos (n = 19), mediana": "{:.4g}",
    "p_FDR": "{:.4f}",
    "Efeito rank-biserial": "{:+.2f}",
})

,"Adultos jovens (n = 35), mediana","Adultos mais velhos (n = 19), mediana",p_FDR,Efeito rank-biserial
metric,,,,
Grau médio por janela (normalizado),0.6063,0.5453,0.0056,-0.49
Variabilidade temporal do grau (DP),0.2002,0.1978,0.0658,-0.32
Agrupamento binário por janela,0.7677,0.7216,0.0056,-0.49
Agrupamento ponderado (REA),0.5911,0.5285,0.0056,-0.49
"Intermediação (REA, distância 1/peso)",7.868e-05,0.0003297,0.0667,+0.31


Com um valor por participante (35 adultos jovens, 19 adultos mais velhos; média de EO e EC), os adultos mais velhos apresentaram, após correção FDR:

- **menor grau médio por janela** (mediana 0,55 vs 0,61; p_FDR = 0,006; rank-biserial = −0,49): em cada janela, os eletrodos se sincronizam com uma fração menor dos demais;
- **menor agrupamento binário por janela** (0,72 vs 0,77; p_FDR = 0,006; r = −0,49) e menor agrupamento ponderado da REA (0,53 vs 0,59; p_FDR = 0,006; r = −0,49).

A variabilidade temporal do grau (DP 0,198 vs 0,200; p_FDR = 0,066) e a intermediação (p_FDR = 0,067) não diferiram entre os grupos na média EO/EC. Ambas diferem em EO (variabilidade menor, p = 0,006; intermediação maior nos adultos mais velhos, p = 0,011), mas esses testes por condição são descritivos.

Uma interpretação para esses resultados é que, com o envelhecimento, as redes funcionais de cada instante (janelas do TVG) ficam mais esparsas e menos agrupadas: cada eletrodo se sincroniza com menos eletrodos por vez, e seus vizinhos formam menos triângulos entre si. A esparsificação vale para as redes por janela; a rede agregada (REA) é completa em todos os participantes e não muda de densidade. A tendência de maior intermediação nos adultos mais velhos, mais clara em EO, sugere que, com menos ligações diretas fortes, alguns eletrodos passam a funcionar como ponte. Esses achados vêm de uma amostra pequena e desbalanceada (19 adultos mais velhos, poucos do sexo feminino) e devem ser lidos como exploratórios.

### Métricas por janela: todos os pontos

Cada ponto é um eletrodo numa janela do TVG (1967 janelas × ~60 eletrodos por registro). Esses pontos servem para ver a distribuição e a variabilidade no tempo, mas não para testes: janelas consecutivas de 100 ms são muito correlacionadas e vêm da mesma pessoa. Os testes acima usam os resumos por participante (média e desvio padrão no tempo).

- **Nuvem:** amostra de 200 pontos por registro, para o gráfico continuar leve.
- **Densidade:** histograma de todos os pontos de cada registro, normalizado e depois tirada a média dentro do grupo (cada registro pesa igual).

In [15]:
from pathlib import Path

WINDOWS_DIR = Path("../../data/intermediate/windows")
POINTS_PER_RECORDING = 200
WINDOW_METRICS = {
    "degree": "Grau por janela (normalizado)",
    "clustering": "Agrupamento binário por janela",
}
# grau por janela é discreto (k / (N − 1), N ≈ 56–61): bins largos evitam serrilhado
bins = np.linspace(0, 1, 21)
rng = np.random.default_rng(42)

recordings = (
    df_healthy[["subject_id", "condition", "Faixa Etária"]]
    .drop_duplicates()
    .rename(columns={"Faixa Etária": "age_group"})
)

cloud_parts, hist_parts = [], []
for rec in recordings.itertuples(index=False):
    candidates = [WINDOWS_DIR / f"{rec.subject_id}_{c}.npz" for c in (rec.condition, "E0")]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        continue
    with np.load(path) as w:
        n_nodes = w["degree"].shape[0]
        series = {"degree": w["degree"] / (n_nodes - 1), "clustering": w["clustering"]}
    for key, label in WINDOW_METRICS.items():
        values = series[key].ravel()
        values = values[np.isfinite(values)]
        counts, _ = np.histogram(values, bins=bins)
        hist_parts.append(pd.DataFrame({
            "metric": label, "condition": rec.condition, "age_group": rec.age_group,
            "subject_id": rec.subject_id, "x": (bins[:-1] + bins[1:]) / 2,
            "densidade": counts / counts.sum() / np.diff(bins),
        }))
        cloud_parts.append(pd.DataFrame({
            "metric": label, "condition": rec.condition, "age_group": rec.age_group,
            "valor": rng.choice(values, size=min(POINTS_PER_RECORDING, len(values)), replace=False),
        }))

window_cloud = pd.concat(cloud_parts, ignore_index=True)
window_hist = pd.concat(hist_parts, ignore_index=True)
print(f"Registros com séries por janela: {window_hist['subject_id'].nunique()} sujeitos, "
      f"{len(hist_parts) // len(WINDOW_METRICS)} registros")

Registros com séries por janela: 54 sujeitos, 108 registros


In [16]:
fig = px.violin(
    window_cloud,
    x="age_group",
    y="valor",
    color="condition",
    facet_col="metric",
    points="all",
    box=True,
    category_orders={
        "age_group": ["young", "older"],
        "condition": ["EO", "EC"],
        "metric": list(WINDOW_METRICS.values()),
    },
    labels={"age_group": "Faixa etária", "valor": "Valor na janela", "condition": "Condição"},
    title="Métricas por janela: eletrodo × janela (amostra por registro)",
)
fig.update_traces(marker=dict(size=2, opacity=0.25), jitter=0.4, pointpos=0, spanmode="hard")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()

In [17]:
group_density = (
    window_hist.groupby(["metric", "condition", "age_group", "x"], as_index=False)["densidade"].mean()
)

fig = px.line(
    group_density,
    x="x",
    y="densidade",
    color="age_group",
    line_dash="condition",
    facet_col="metric",
    category_orders={
        "age_group": ["young", "older"],
        "condition": ["EO", "EC"],
        "metric": list(WINDOW_METRICS.values()),
    },
    labels={"x": "Valor na janela", "densidade": "Densidade", "age_group": "Faixa etária", "condition": "Condição"},
    title="Distribuição das métricas por janela (média das densidades dos registros)",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(matches=None)
fig.show()

Gênero

In [18]:
df_healthy["Gênero"].value_counts()

Gênero
2    84
1    24
Name: count, dtype: int64

In [19]:
df_healthy["Gênero"].dtype
df_healthy["Gênero"].unique()


array(['2', '1'], dtype=object)

In [20]:
df_healthy["Gênero"] = df_healthy["Gênero"].astype(float).astype(int)

df_healthy["gender_bin"] = df_healthy["Gênero"].map({
    1: 0,  # female
    2: 1   # male
})

df_healthy["gender_bin"].value_counts()


gender_bin
1    84
0    24
Name: count, dtype: int64

In [21]:
gender_results = []

for metric in GRAPH_METRICS:
    valid = df_healthy[[metric, "gender_bin"]].dropna()
    r, p = pointbiserialr(valid["gender_bin"], valid[metric])
    gender_results.append({
        "metric": metric,
        "point_biserial_r": r,
        "p_value": p
    })

pd.DataFrame(gender_results)



,metric,point_biserial_r,p_value
0,degree_mean,-0.056412,0.561987
1,sd_degree_mean,-0.052552,0.589090
2,bin_clustering_mean,-0.066671,0.492988
3,clustering_mean,-0.056425,0.561900
4,betweenness_mean,0.076523,0.431195


### Comparação por gênero das métricas de rede

Codificação do metadado: `1 = feminino`, `2 = masculino`.

EO e EC vêm do mesmo sujeito, então os testes são feitos **separadamente por condição** (uma linha por sujeito), com Mann-Whitney bilateral e correção de Benjamini–Hochberg sobre os 10 testes (5 métricas × 2 condições). O tamanho de efeito é a correlação rank-biserial: valores positivos indicam métrica maior no grupo feminino.

O grupo feminino é pequeno, então a comparação tem baixo poder estatístico.

In [22]:
gender_label_map = {1: "Feminino", 2: "Masculino"}
gender_order = ["Feminino", "Masculino"]

df_healthy["Gênero_label"] = (
    pd.to_numeric(df_healthy["Gênero"]).astype(int).map(gender_label_map)
)

# sujeitos por gênero e faixa etária (checa se idade confunde a comparação)
(
    df_healthy.drop_duplicates("subject_id")
    .groupby(["Faixa Etária", "Gênero_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=gender_order)
)

Gênero_label,Feminino,Masculino
Faixa Etária,,
older,4,15
young,8,27


In [23]:
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Gênero_label",
        y=metric,
        color="Gênero_label",
        points="all",
        category_orders={"Gênero_label": gender_order},
        labels={"Gênero_label": "Gênero", metric: label},
    )
    fig.update_layout(
        xaxis_title="Gênero",
        yaxis_title=label,
        title=f"{label} vs Gênero"
    )
    fig.show()

In [24]:
# Boxplot Gênero x Condition
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Gênero_label",
        y=metric,
        color="condition",
        points="all",
        category_orders={"Gênero_label": gender_order, "condition": ["EO", "EC"]},
        labels={"Gênero_label": "Gênero", "condition": "Condição", metric: label},
    )
    fig.update_layout(
        xaxis_title="Gênero",
        yaxis_title=label,
        title=f"{label} vs Gênero (EO e EC)"
    )
    fig.show()

In [25]:
# Boxplot Gênero x Faixa Etária, separado por condição
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Faixa Etária",
        y=metric,
        color="Gênero_label",
        facet_col="condition",
        points="all",
        category_orders={
            "Faixa Etária": ["young", "older"],
            "Gênero_label": gender_order,
            "condition": ["EO", "EC"],
        },
        labels={"Gênero_label": "Gênero", "condition": "Condição", metric: label},
    )
    fig.update_layout(title=f"{label} vs Gênero por Faixa Etária")
    fig.show()

In [26]:
from scipy.stats import false_discovery_control, mannwhitneyu

gender_mw_rows = []
for cond in ["EO", "EC"]:
    sub = df_healthy[df_healthy["condition"] == cond]
    for metric, label in GRAPH_METRICS.items():
        fem = sub.loc[sub["Gênero_label"] == "Feminino", metric].dropna()
        masc = sub.loc[sub["Gênero_label"] == "Masculino", metric].dropna()
        u, p = mannwhitneyu(fem, masc, alternative="two-sided")
        gender_mw_rows.append({
            "condition": cond,
            "metric": label,
            "n_fem": len(fem),
            "n_masc": len(masc),
            "mediana_fem": fem.median(),
            "mediana_masc": masc.median(),
            "U": u,
            "p_value": p,
            # positivo = feminino maior
            "rank_biserial": 2 * u / (len(fem) * len(masc)) - 1,
        })

gender_mw = pd.DataFrame(gender_mw_rows)
gender_mw["p_fdr"] = false_discovery_control(gender_mw["p_value"])
gender_mw

,condition,metric,n_fem,n_masc,mediana_fem,mediana_masc,U,p_value,rank_biserial,p_fdr
0,EO,Grau médio por janela (normalizado),12,42,0.582218,0.544541,286.0,0.485797,0.134921,0.853781
1,EO,Variabilidade temporal do grau (DP),12,42,0.201835,0.199038,282.0,0.539358,0.119048,0.853781
2,EO,Agrupamento binário por janela,12,42,0.756176,0.721571,296.0,0.365427,0.174603,0.853781
3,EO,Agrupamento ponderado (REA),12,42,0.564550,0.525470,286.0,0.485797,0.134921,0.853781
4,EO,"Intermediação (REA, distância 1/peso)",12,42,0.000168,0.000147,231.0,0.667998,-0.083333,0.853781
5,EC,Grau médio por janela (normalizado),12,42,0.621612,0.586566,277.0,0.610224,0.099206,0.853781
6,EC,Variabilidade temporal do grau (DP),12,42,0.199453,0.199997,260.0,0.875996,0.031746,0.875996
7,EC,Agrupamento binário por janela,12,42,0.774374,0.748660,275.0,0.639684,0.091270,0.853781
8,EC,Agrupamento ponderado (REA),12,42,0.609071,0.573986,269.0,0.731370,0.067460,0.853781
9,EC,"Intermediação (REA, distância 1/peso)",12,42,0.000097,0.000061,266.5,0.768403,0.057540,0.853781


### Interação idade × condição

Diferença EO − EC de cada participante, comparada entre adultos mais velhos e adultos jovens. Só com essa diferença significativa dá para dizer que o efeito da idade muda entre olhos abertos e fechados.

In [27]:
interaction_tests = pd.DataFrame(interaction_rows)
interaction_tests["p_fdr"] = false_discovery_control(interaction_tests["p_value"])
interaction_tests.round(4)

,metric,medida,n_older,n_young,mediana_older,mediana_young,p_value,rank_biserial,p_fdr
0,Grau médio por janela (normalizado),EO − EC,19,35,-0.0702,-0.0116,0.0160,-0.4015,0.0255
1,Variabilidade temporal do grau (DP),EO − EC,19,35,-0.0071,0.0029,0.0356,-0.3504,0.0356
2,Agrupamento binário por janela,EO − EC,19,35,-0.0571,-0.0112,0.0138,-0.4105,0.0255
3,Agrupamento ponderado (REA),EO − EC,19,35,-0.0735,-0.0128,0.0204,-0.3865,0.0255
4,"Intermediação (REA, distância 1/peso)",EO − EC,19,35,0.0001,0.0000,0.0041,0.4767,0.0206


A diferença EO − EC depende da idade em todas as métricas (p_FDR ≤ 0,036). Ao abrir os olhos, o grau e o agrupamento caem mais nos adultos mais velhos (mediana EO − EC do grau: −0,070 vs −0,012 nos adultos jovens), a variabilidade do grau cai nos adultos mais velhos e fica estável nos adultos jovens, e a intermediação sobe nos adultos mais velhos. Por isso as diferenças entre grupos etários são maiores em EO do que em EC. Mesmo em indivíduos saudáveis, o envelhecimento aparece associado a redes instantâneas mais esparsas e menos agrupadas, com uma resposta mais forte à abertura dos olhos.